This notebook is used to train a classifier that takes as features the ones extracted from Neural networks
and then train a cassifier on the top pf them 

In [1]:
data_folder="../Data/"

import sys
import os

sys.path.append(os.path.abspath("../Codes"))

sys.path.append(os.path.abspath("../_Deep_Learning"))

from Utils import freq_filter, windowize,  RegionWiseStandardizer

from BCI_Library import read_subject, saveresults_pickle, compute_welch
from BCI_Library import  train_single_model,select_regions_Cohen_effect_size, build_compiled_mlp

2026-03-02 09:49:27.426892: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 09:49:27.482685: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 09:49:28.641119: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Region Agnostic - test on the same subject

In [ ]:
import os
import sys
import tensorflow.keras as keras

# To be changed
Tutti_soggetti = [2]
DataType = "EEG"
CNN_o_Autoencoder = "Autoencoder" # Autoencoder, # CNN
model_path_4 = "2026-02-27-10_37_22_Autoencoder_1region_region_agnostic_First_Try_No_BatchNorm" + "/"

# Fixed
model_path_1 = f"../_Deep_Learning/Models_trained/Subject_"
# model path 2 = f"{subject_index}/"
# model path 2.5 = f"{DataType}/"
model_path_3 = f"{CNN_o_Autoencoder}" + ("s/" if CNN_o_Autoencoder=="Autoencoder" else "/")

lib_path = model_path_1 + f"{Tutti_soggetti[0]}/" + f"{DataType}/" + model_path_3 + model_path_4

sys.path.append(os.path.abspath(lib_path))

try :
    from _Backup_Library import MultiTaskModel
except:
    from Deep_Library_BCI import MultiTaskModel
    raise Exception("Non ho caricato dal backup, controlla bene se è quello che vuoi")
    print("\n\n########################## ATTENTO  ##########################\n non ho caricato dal backup\n\n")


if CNN_o_Autoencoder == "CNN":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=True, use_decoder=False)
if CNN_o_Autoencoder == "Autoencoder":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)

model.build(input_shape=(128,1))

8 128


## Fixed region selection 

In [5]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]

In [13]:
Cohen_global_EEG_8 = [14, 32, 33, 44, 46, 48, 50, 62]
Cohen_global_EEG_10 = [14, 20, 32, 33, 44, 46, 48, 50, 58, 62]


Cohen_global_MEG_8 = [6, 32, 42, 43, 44, 46, 48, 51]
Cohen_global_MEG_10 = [14, 32, 42, 43, 44, 46, 47, 48, 51, 62]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Global Selection - Cohen based" #"Global Selection - Cohen based" # "-" # Yes Important Regions - all
regions_selected = Cohen_global_MEG_8 #Cohen_global_MEG_8 #Cohen_global_EEG_8 #Interesting_regions

Classifier_names, meths =  ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}


PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)

tutte_roi = 68

verbose = False


for Classifier_name, meth in zip(Classifier_names,meths):

    Rows = pd.DataFrame({
        "ROIs": [],
        "NROIs":[],
        "DataType": [],
        "freqs_band": [],  
        "subject": [],
        "Classifier": [],
        "Accuracy": [],
        "Features": [],
        "Comments": [],
        "Region Selection": [],
        "Date": []
    })
    
    print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
    for subject_index in tqdm.tqdm(Tutti_soggetti):

        model_path = model_path_1 + f"{subject_index}/" + f"{DataType}/" + model_path_3 + model_path_4 
        Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

        filepath = filepath_ori + f"{subject_index}.pkl"
        file_input = filepath
        file_output = filepath

        data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
        data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
        data = np.concatenate((data_2_Rest, data_2_MI), axis=0)

        data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

        # y shape: (n_trials,) with negative values for Rest and positive for MI
        y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

        
        fold_accuracies = []
        for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
        ########################## DATA PREPROCESSING ##########################################
        # TODO create (num_windows,0)
            Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
            Feat_test_regions_selected = None # np.empty()

            
            for region_index, regione in enumerate(regions_selected):
                
                split_num = fold_idx+1
                model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
    

                X_train = data[train_idx,[regione]]
                y_train = y[train_idx]
                # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                X_Block = data[block_idx,[regione]]
                y_Block = y[block_idx]


                X_val, X_test, y_val, y_test = train_test_split(
                X_Block, y_Block, test_size=0.5,         
                shuffle=True, stratify=y_Block,     # preserves label balance
                random_state=random_seed)

                y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                # assegno valore in base a ROI, da ±1 a ±68
                y_train[:, 0] *= regione+1
                
                # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                y_val[:, 0] *= regione+1

                y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                y_test[:, 0] *= regione+1


                x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                x_val = X_val.reshape(-1, X_val.shape[-1])
                x_test = X_test.reshape(-1, X_test.shape[-1])

                y_train = y_train.reshape(-1)
                y_val = y_val.reshape(-1)
                y_test = y_test.reshape(-1)

                # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

            
                #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                ###################################################################################################################################
                # NORMALIZATION (per region?)
                # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                scaler = RegionWiseStandardizer()
                x_train = scaler.fit_transform(x_train, y_train)
                x_val = scaler.transform(x_val, y_val)
                x_test = scaler.transform(x_test, y_test)
                # scaler = TraceWiseStandardizer()
                # x_train = scaler.transform(x_train)
                # x_val = scaler.transform(x_val)
                # x_test = scaler.transform(x_test)
                x_train = x_train[..., np.newaxis]  
                x_val = x_val[..., np.newaxis]  
                x_test = x_test[..., np.newaxis]
                # x_train.shape (n_windows, timepoints, 1)
                
                Features_training_1_regione = model.encoder(x_train)
                Features_test_1_regione = model.encoder(x_test)
                # Features_training_1_regione  shape (n_windows,16)
                
                
                ################################### Feature Extractions ##########################################
                if Feat_train_regions_selected is None:
                    Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                    Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


            Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
            # Feat_.shape (Trials, selected_ROIs, features for Roi)
            labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

            train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
            val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


            if Classifier_name.lower()=="mlp":
                    meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                    
            acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

            fold_accuracies.append(acc)
            if verbose:
                print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                print(cm)

            #####################################################################################################################

        if verbose:
            print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
            print("Std accuracy:", np.std(fold_accuracies))

        new_row = {
            "ROIs": [regions_selected],
            "NROIs":[len(regions_selected)],
            "DataType": [DataType],
            "freqs_band": ["-"],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Accuracy": [fold_accuracies],
            "Features": [Features_name],
            "Comments": [Comments],
            "Region Selection": [Region_Selection],
            "Date": [today],
            "Personalized_Freq_Range": ["-"],
        }
        Rows=pd.concat((Rows,pd.DataFrame(new_row)))
    

    if saveresults:
        saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 MEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:09<00:00,  9.90s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 MEG SVC


100%|██████████| 1/1 [00:08<00:00,  8.35s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 MEG RandomForest


100%|██████████| 1/1 [00:12<00:00, 12.90s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


In [29]:
import pandas as pd
leggi  ="../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_1.pkl"
uu = pd.read_pickle(leggi)

In [30]:
indeces_drop = list( uu[ (uu["DataType"]=="MEG") * (uu["Region Selection"]=="Global Selection - Cohen based") ].index ) 
uu[ (uu["DataType"]=="MEG") * (uu["Region Selection"]=="Global Selection - Cohen based") ]

,ROIs,NROIs,DataType,freqs_band,subject,Classifier,Accuracy,Features,Region Selection,Date,Personalized_Freq_Range,Comments
30,"[6, 32, 42, 43, 44, 46, 48, 51]",8.0,MEG,-,1.0,LDA,"[0.569377990430622, 0.6555023923444976, 0.6172...",CNN_extracted,Global Selection - Cohen based,2026-02-15,-,2026-02-11-12_21_37_CNN_1region_region_agnosti...
31,"[6, 32, 42, 43, 44, 46, 48, 51]",8.0,MEG,-,1.0,SVC,"[0.5502392344497608, 0.6650717703349283, 0.641...",CNN_extracted,Global Selection - Cohen based,2026-02-15,-,2026-02-11-12_21_37_CNN_1region_region_agnosti...
32,"[6, 32, 42, 43, 44, 46, 48, 51]",8.0,MEG,-,1.0,RandomForest,"[0.5311004784688995, 0.6507177033492823, 0.593...",CNN_extracted,Global Selection - Cohen based,2026-02-15,-,2026-02-11-12_21_37_CNN_1region_region_agnosti...


In [25]:
indeces_drop = [33,34,35]
indeces_drop

[33, 34, 35]

In [26]:
uu.drop(indeces_drop,inplace=True)
uu.reset_index(inplace=True,drop=True)

In [ ]:
# uu.to_pickle(leggi)

In [26]:
import pandas as pd
set(pd.read_pickle("../Results/BCI_Performances/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl")["Region Selection"])

{"Cohen's d effect size selection",
 "Cohen's d effect size selection spectra starting from 3s",
 'Global Selection - Cohen based',
 'Global Selection - MyCriteria based',
 'MyCriteria selection',
 'MyCriteria2 selection',
 'T test selection mean power alpha beta'}

## Per fold region selection

In [3]:
def compute_power_spectra(data_2_MI, data_2_Rest):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]})

    return WMI, WRe


def selezione_cohen(WRe, WMI, train_idx, num_best_ROIs):
    training_index_Re = train_idx[np.where(train_idx<len(WRe))] 
    training_index_MI = train_idx[np.where(train_idx>=len(WRe))]-len(WRe)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_Re], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Cohen's d effect size selection" # "-"

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


num_best_ROIs = 8


verbose = False

Classifier_names, meths =  ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}
DataTypes = ["EEG"]

for DataType in DataTypes:
    for Classifier_name, meth in zip(Classifier_names,meths):
        
        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(Tutti_soggetti):

            model_path = model_path_1 + f"{subject_index}/" + f"{DataType}/" + model_path_3 + model_path_4
            Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

            filepath = filepath_ori + f"{subject_index}.pkl"
            file_input = filepath
            file_output = filepath

            data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
            data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
            data = np.concatenate((data_2_Rest, data_2_MI), axis=0)
            
            WMI, WRe = compute_power_spectra(data_2_MI,data_2_Rest)
            
            data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
            data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

            # y shape: (n_trials,) with negative values for Rest and positive for MI
            y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

            
            fold_accuracies = []
            tutti_fold_selezioni_regioni = []
            for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
            ########################## DATA PREPROCESSING ##########################################
            # TODO create (num_windows,0)
                Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
                Feat_test_regions_selected = None # np.empty()
                
                regions_selected = selezione_cohen(WRe, WMI, train_idx, num_best_ROIs)
                regions_selected.sort()
                tutti_fold_selezioni_regioni.append(regions_selected)
                
                for region_index, regione in enumerate(regions_selected):
                   
                    split_num = fold_idx+1
                    model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
        

                    X_train = data[train_idx,[regione]]
                    y_train = y[train_idx]
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                    X_Block = data[block_idx,[regione]]
                    y_Block = y[block_idx]


                    X_val, X_test, y_val, y_test = train_test_split(
                    X_Block, y_Block, test_size=0.5,         
                    shuffle=True, stratify=y_Block,     # preserves label balance
                    random_state=random_seed)

                    y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                    # assegno valore in base a ROI, da ±1 a ±68
                    y_train[:, 0] *= regione+1
                    
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                    y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                    y_val[:, 0] *= regione+1

                    y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                    y_test[:, 0] *= regione+1


                    x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                    x_val = X_val.reshape(-1, X_val.shape[-1])
                    x_test = X_test.reshape(-1, X_test.shape[-1])

                    y_train = y_train.reshape(-1)
                    y_val = y_val.reshape(-1)
                    y_test = y_test.reshape(-1)

                    # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                    # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                    x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                    x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                    x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                    # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                    # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

                
                    #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                    ###################################################################################################################################
                    # NORMALIZATION (per region?)
                    # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                    # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                    # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                    scaler = RegionWiseStandardizer()
                    x_train = scaler.fit_transform(x_train, y_train)
                    x_val = scaler.transform(x_val, y_val)
                    x_test = scaler.transform(x_test, y_test)
                    # scaler = TraceWiseStandardizer()
                    # x_train = scaler.transform(x_train)
                    # x_val = scaler.transform(x_val)
                    # x_test = scaler.transform(x_test)
                    x_train = x_train[..., np.newaxis]  
                    x_val = x_val[..., np.newaxis]  
                    x_test = x_test[..., np.newaxis]
                    # x_train.shape (n_windows, timepoints, 1)
                    
                    Features_training_1_regione = model.encoder(x_train)
                    Features_test_1_regione = model.encoder(x_test)
                    # Features_training_1_regione  shape (n_windows,16)
                    
                    
                    ################################### Feature Extractions ##########################################
                    if Feat_train_regions_selected is None:
                        Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                        Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                    Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                    Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


                Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
                # Feat_.shape (Trials, selected_ROIs, features for Roi)
                labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

                train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
                val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


                if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": ["-"],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Accuracy": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": ["-"],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:12<00:00, 12.54s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


##############################
 EEG SVC


100%|██████████| 1/1 [00:11<00:00, 11.24s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


##############################
 EEG RandomForest


100%|██████████| 1/1 [00:18<00:00, 18.50s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl


In [21]:
tutti_fold_selezioni_regioni

[[15, 20, 26, 27, 34, 50, 51, 58],
 [7, 20, 26, 27, 34, 35, 50, 51],
 [0, 7, 15, 26, 27, 34, 50, 51],
 [20, 26, 27, 34, 35, 50, 51, 58],
 [12, 20, 21, 26, 27, 34, 50, 51]]

In [15]:
import pandas as pd
uu = pd.read_pickle("../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl")

In [16]:
uu.iloc[[-1,-2,-3]].Accuracy.apply(np.mean)

51    0.733876
50    0.765694
49    0.635598
Name: Accuracy, dtype: float64

In [8]:
set(uu["Region Selection"])


{"Cohen's d effect size selection",
 'Global Selection - Cohen based',
 'Yes Important Regions - all'}

# Region Specific - test on the same subject

In [ ]:
import os
import sys
import tensorflow.keras as keras

# To be changed
Tutti_soggetti = [8]
DataType = "EEG"
CNN_o_Autoencoder = "Autoencoder" # Autoencoder, # CNN
model_path_4 = "2026-02-17-11_11_35_Autoencoder_1region_region_specific_cohen_subject_specific" + "/"

# Fixed
model_path_1 = f"../_Deep_Learning/Models_trained/Subject_"
# model path 2 = f"{subject_index}/"
# model path 2.5 = f"{DataType}/"
model_path_3 = f"{CNN_o_Autoencoder}" + ("s/" if CNN_o_Autoencoder=="Autoencoder" else "/")

lib_path = model_path_1 + f"{Tutti_soggetti[0]}/" + f"{DataType}/" + model_path_3 + model_path_4

sys.path.append(os.path.abspath(lib_path))

try :
    from _Backup_Library import MultiTaskModel
except:
    from Deep_Library_BCI import MultiTaskModel
    #raise Exception("Non ho caricato dal backup, controlla bene se è quello che vuoi")
    print("\n\n########################## ATTENTO  ##########################\n non ho caricato dal backup\n\n")


if CNN_o_Autoencoder == "CNN":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=True, use_decoder=False)
if CNN_o_Autoencoder == "Autoencoder":
    model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)

model.build(input_shape=(128,1))

Exception: Non ho caricato dal backup, controlla bene se è quello che vuoi

## Per fold region selection

In [3]:
def compute_power_spectra(data_2_MI, data_2_Rest):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]})

    return WMI, WRe


def selezione_cohen(WRe, WMI, train_idx, num_best_ROIs):
    training_index_Re = train_idx[np.where(train_idx<len(WRe))] 
    training_index_MI = train_idx[np.where(train_idx>=len(WRe))]-len(WRe)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_Re], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Cohen's d effect size selection" # "-"

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


num_best_ROIs = 8


verbose = False

Classifier_names, meths =  ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}
DataTypes = ["EEG"]

for Classifier_name, meth in zip(Classifier_names,meths):
    
    Rows = pd.DataFrame({
        "ROIs": [],
        "NROIs":[],
        "DataType": [],
        "freqs_band": [],  
        "subject": [],
        "Classifier": [],
        "Accuracy": [],
        "Features": [],
        "Comments": [],
        "Region Selection": [],
        "Date": []
    })
    
    print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
    for subject_index in tqdm.tqdm(Tutti_soggetti):

        model_path = model_path_1 + f"{subject_index}/" + f"{DataType}/" + model_path_3 + model_path_4
        Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

        filepath = filepath_ori + f"{subject_index}.pkl"
        file_input = filepath
        file_output = filepath

        data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
        data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
        data = np.concatenate((data_2_Rest, data_2_MI), axis=0)
        
        WMI, WRe = compute_power_spectra(data_2_MI,data_2_Rest)
        
        data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

        # y shape: (n_trials,) with negative values for Rest and positive for MI
        y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

        
        fold_accuracies = []
        tutti_fold_selezioni_regioni = []
        for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
        ########################## DATA PREPROCESSING ##########################################
        # TODO create (num_windows,0)
            Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
            Feat_test_regions_selected = None # np.empty()
            
            regions_selected = selezione_cohen(WRe, WMI, train_idx, num_best_ROIs)
            regions_selected.sort()
            tutti_fold_selezioni_regioni.append(regions_selected)
            
            for region_index, regione in enumerate(regions_selected):
                
                split_num = fold_idx+1
                model.load_weights(model_path+f"Split_{split_num}/Region_{regione}/model.weights.h5")

                X_train = data[train_idx,[regione]]
                y_train = y[train_idx]
                # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                X_Block = data[block_idx,[regione]]
                y_Block = y[block_idx]

                X_val, X_test, y_val, y_test = train_test_split(
                X_Block, y_Block, test_size=0.5,         
                shuffle=True, stratify=y_Block,     # preserves label balance
                random_state=random_seed)

                y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                # assegno valore in base a ROI, da ±1 a ±68
                y_train[:, 0] *= regione+1
                
                # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                y_val[:, 0] *= regione+1

                y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                y_test[:, 0] *= regione+1


                x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                x_val = X_val.reshape(-1, X_val.shape[-1])
                x_test = X_test.reshape(-1, X_test.shape[-1])

                y_train = y_train.reshape(-1)
                y_val = y_val.reshape(-1)
                y_test = y_test.reshape(-1)

                # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

            
                #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                ###################################################################################################################################
                # NORMALIZATION (per region?)
                # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                scaler = RegionWiseStandardizer()
                x_train = scaler.fit_transform(x_train, y_train)
                x_val = scaler.transform(x_val, y_val)
                x_test = scaler.transform(x_test, y_test)
                # scaler = TraceWiseStandardizer()
                # x_train = scaler.transform(x_train)
                # x_val = scaler.transform(x_val)
                # x_test = scaler.transform(x_test)
                x_train = x_train[..., np.newaxis]  
                x_val = x_val[..., np.newaxis]  
                x_test = x_test[..., np.newaxis]
                # x_train.shape (n_windows, timepoints, 1)
                
                Features_training_1_regione = model.encoder(x_train)
                Features_test_1_regione = model.encoder(x_test)
                # Features_training_1_regione  shape (n_windows,16)
                
                
                ################################### Feature Extractions ##########################################
                if Feat_train_regions_selected is None:
                    Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                    Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


            Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
            # Feat_.shape (Trials, selected_ROIs, features for Roi)
            labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

            train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
            val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


            if Classifier_name.lower()=="mlp":
                    meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                    
            acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

            fold_accuracies.append(acc)
            if verbose:
                print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                print(cm)

            #####################################################################################################################

        if verbose:
            print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
            print("Std accuracy:", np.std(fold_accuracies))

        new_row = {
            "ROIs": [regions_selected],
            "NROIs":[len(regions_selected)],
            "DataType": [DataType],
            "freqs_band": ["-"],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Accuracy": [fold_accuracies],
            "Features": [Features_name],
            "Comments": [Comments],
            "Region Selection": [Region_Selection],
            "Date": [today],
            "Personalized_Freq_Range": ["-"],
        }
        Rows=pd.concat((Rows,pd.DataFrame(new_row)))
    

    if saveresults:
        saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-17 12:41:40.659678: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.48GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-02-17 12:41:42.499668: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.50GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
100%|██████████| 1/1 [00:13<00:00, 13.65s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 EEG SVC


100%|██████████| 1/1 [00:11<00:00, 11.69s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


##############################
 EEG RandomForest


100%|██████████| 1/1 [00:16<00:00, 16.53s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_8.pkl


In [ ]:
tutti_fold_selezioni_regioni

[[15, 20, 26, 27, 34, 50, 51, 58],
 [7, 20, 26, 27, 34, 35, 50, 51],
 [0, 7, 15, 26, 27, 34, 50, 51],
 [20, 26, 27, 34, 35, 50, 51, 58],
 [12, 20, 21, 26, 27, 34, 50, 51]]

In [ ]:
import pandas as pd
uu = pd.read_pickle("../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_2.pkl")

In [ ]:
set(uu["Region Selection"])


{"Cohen's d effect size selection",
 'Global Selection - Cohen based',
 'Yes Important Regions - all'}

# Region Agnostic - test on the same subject - splitted frequencies

In [ ]:
import os
import sys
import tensorflow.keras as keras
import re

def extract_pairs(s):
    # Find the last underscore section before "_Hz" or "_hz"
    match = re.search(r'_((?:\d+-\d+_)*\d+-\d+)_?hz/$', s, flags=re.IGNORECASE)
    if not match:
        return []
    segment = match.group(1)  # e.g. "6-13_12-21_19-31"
    pairs = re.findall(r'(\d+)-(\d+)', segment)
    return [(int(a), int(b)) for a, b in pairs]

# To be changed
Tutti_soggetti = [18]
DataType = "EEG"
CNN_o_Autoencoder = "Autoencoder" # Autoencoder, # CNN
model_path_4 = "2026-02-27-11_42_19_Autoencoder_1region_region_agnostic_separate_frequencies_6-13_12-21_19-31_Hz" + "/"
num_bands = 3

separate_frequencies = extract_pairs(model_path_4)

if len(separate_frequencies) != num_bands:
    raise ValueError(f"Expected {num_bands} frequency bands, but extracted {len(separate_frequencies)} from the model path.")


# Fixed
model_path_1 = f"../_Deep_Learning/Models_trained/Subject_"
# model path 2 = f"{subject_index}/"
# model path 2.5 = f"{DataType}/"
model_path_3 = f"{CNN_o_Autoencoder}" + ("s/" if CNN_o_Autoencoder=="Autoencoder" else "/")

lib_path = model_path_1 + f"{Tutti_soggetti[0]}/" + f"{DataType}/" + model_path_3 + model_path_4

sys.path.append(os.path.abspath(lib_path))

try :
    from _Backup_Library import MultiTaskModel
except:
    from Deep_Library_BCI import MultiTaskModel
    raise Exception("Non ho caricato dal backup, controlla bene se è quello che vuoi")
    print("\n\n########################## ATTENTO  ##########################\n non ho caricato dal backup\n\n")


if CNN_o_Autoencoder == "CNN":
    model = MultiTaskModel(input_shape=(128,num_bands), use_classifier=True, use_decoder=False)
if CNN_o_Autoencoder == "Autoencoder":
    model = MultiTaskModel(input_shape=(128,num_bands), use_classifier=False, use_decoder=True)

model.build(input_shape=(128,num_bands))

8 128


## Fixed region selection 

In [ ]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]

In [ ]:
Cohen_global_EEG_8 = [14, 32, 33, 44, 46, 48, 50, 62]
Cohen_global_EEG_10 = [14, 20, 32, 33, 44, 46, 48, 50, 58, 62]


Cohen_global_MEG_8 = [6, 32, 42, 43, 44, 46, 48, 51]
Cohen_global_MEG_10 = [14, 32, 42, 43, 44, 46, 47, 48, 51, 62]

## Per fold region selection

In [3]:
def compute_power_spectra(data_2_MI, data_2_Rest):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]})

    return WMI, WRe


def selezione_cohen(WRe, WMI, train_idx, num_best_ROIs):
    training_index_Re = train_idx[np.where(train_idx<len(WRe))] 
    training_index_MI = train_idx[np.where(train_idx>=len(WRe))]-len(WRe)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_Re], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Cohen's d effect size selection" # "-"

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


num_best_ROIs = 8


verbose = False

Classifier_names, meths = ["LDA", "SVC", "RandomForest"], [LDA, SVC, RandomForestClassifier] # ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}
DataTypes = ["EEG"]


for Classifier_name, meth in zip(Classifier_names,meths):
    
    Rows = pd.DataFrame({
        "ROIs": [],
        "NROIs":[],
        "DataType": [],
        "freqs_band": [],  
        "subject": [],
        "Classifier": [],
        "Accuracy": [],
        "Features": [],
        "Comments": [],
        "Region Selection": [],
        "Date": []
    })
    
    print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
    for subject_index in tqdm.tqdm(Tutti_soggetti):

        model_path = model_path_1 + f"{subject_index}/" + f"{DataType}/" + model_path_3 + model_path_4
        Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

        filepath = filepath_ori + f"{subject_index}.pkl"
        file_input = filepath
        file_output = filepath

        data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
        data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
        data = np.concatenate((data_2_Rest, data_2_MI), axis=0)
        
        WMI, WRe = compute_power_spectra(data_2_MI,data_2_Rest)
        
        X = []
        for start_freq, end_freq in separate_frequencies:
            X.append(freq_filter(data, sf=250, freqs=[start_freq,end_freq], type_filter="bandpass") ) # axis = -1 by default 
        data = np.stack(X, axis=-1)         # shape (n_trials, n_regions, n_timepoints, n_freq_bands)
        data = data[:,:,start:end,:]        #  data.shape = (192, 68, 748, freq_bands)

        # y shape: (n_trials,) with negative values for Rest and positive for MI
        y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

        
        fold_accuracies = []
        tutti_fold_selezioni_regioni = []
        for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
        ########################## DATA PREPROCESSING ##########################################
        # TODO create (num_windows,0)
            Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
            Feat_test_regions_selected = None # np.empty()
            
            regions_selected = selezione_cohen(WRe, WMI, train_idx, num_best_ROIs)
            regions_selected.sort()
            tutti_fold_selezioni_regioni.append(regions_selected)
            
            for region_index, regione in enumerate(regions_selected):
                
                split_num = fold_idx+1
                model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
    

                X_train = data[train_idx,[regione]]
                y_train = y[train_idx]

                # X_Train shape: (n_trials_train, n_timepoints, freq_bands) (single region)
                # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                X_Block = data[block_idx,[regione]]
                y_Block = y[block_idx]


                X_val, X_test, y_val, y_test = train_test_split(
                X_Block, y_Block, test_size=0.5,         
                shuffle=True, stratify=y_Block,     # preserves label balance
                random_state=random_seed)

                y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                # assegno valore in base a ROI, da ±1 a ±68
                y_train[:, 0] *= regione+1
                
                # X_Train shape: (n_trials_train, n_timepoints, freq_bands)
                # y_Train shape: (n_trials_train, ∈ {-68,...-1,+1,...,+68}

                y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                y_val[:, 0] *= regione+1

                y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                y_test[:, 0] *= regione+1


                x_train = X_train.reshape(-1, X_train.shape[-2],  X_train.shape[-1]) # shape (n_trials*n_regions, n_timepoints, freq_bands)
                x_val = X_val.reshape(-1, X_val.shape[-2],  X_val.shape[-1])
                x_test = X_test.reshape(-1, X_test.shape[-2],  X_test.shape[-1])

                y_train = y_train.reshape(-1)
                y_val = y_val.reshape(-1)
                y_test = y_test.reshape(-1)

                # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                x_trains = []; x_vals = []; x_tests = []

                for num_freq in range(x_train.shape[-1]):
                    x_train_freq, Y_train = windowize(x_train[:,:, num_freq], y_train, win_len=input_shape, shift=shift)
                    x_val_freq, Y_val     = windowize(x_val[:,:, num_freq], y_val, win_len=input_shape, shift=shift)
                    x_test_freq, Y_test   = windowize(x_test[:,:, num_freq], y_test, win_len=input_shape, shift=shift)
                    x_trains.append(x_train_freq);  x_vals.append(x_val_freq);  x_tests.append(x_test_freq); 
                x_train = np.stack(x_trains, axis=-1)
                x_val = np.stack(x_vals, axis=-1)
                x_test = np.stack(x_tests, axis=-1)
                y_train = Y_train; y_val = Y_val; y_test = Y_test
                # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)


                ###################################################################################################################################
                # NORMALIZATION (per region?)
                # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                for num_freq in range(x_train.shape[-1]):
                    scaler = RegionWiseStandardizer()
                    # fit requires a (n_samples, n_timepoints) shaped-array
                    x_train[:,:, num_freq] = scaler.fit_transform(x_train[:,:, num_freq], y_train)
                    x_val[:,:, num_freq] = scaler.transform(x_val[:,:, num_freq], y_val)
                    x_test[:,:, num_freq] = scaler.transform(x_test[:,:, num_freq], y_test)
                    # scaler = TraceWiseStandardizer()
                    # x_train[:,:, num_freq] = scaler.transform(x_train[:,:, num_freq])
                    # x_val[:,:, num_freq] = scaler.transform(x_val[:,:, num_freq])
                    # x_test[:,:, num_freq] = scaler.transform(x_test[:,:, num_freq])
                # x_train = x_train[..., np.newaxis]  
                # x_val = x_val[..., np.newaxis]  
                # x_test = x_test[..., np.newaxis] 
                # x_train.shape (n_windows, timepoints, 1)

                Features_training_1_regione = model.encoder(x_train)
                Features_test_1_regione = model.encoder(x_test)
                # Features_training_1_regione  shape (n_windows,16)
                
                
                ################################### Feature Extractions ##########################################
                if Feat_train_regions_selected is None:
                    Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                    Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


            Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
            # Feat_.shape (Trials, selected_ROIs, features for Roi)
            labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

            train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
            val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


            if Classifier_name.lower()=="mlp":
                    meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                    
            acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

            fold_accuracies.append(acc)
            if verbose:
                print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                print(cm)

            #####################################################################################################################

        if verbose:
            print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
            print("Std accuracy:", np.std(fold_accuracies))

        new_row = {
            "ROIs": [regions_selected],
            "NROIs":[len(regions_selected)],
            "DataType": [DataType],
            "freqs_band": ["-"],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Accuracy": [fold_accuracies],
            "Features": [Features_name],
            "Comments": [Comments],
            "Region Selection": [Region_Selection],
            "Date": [today],
            "Personalized_Freq_Range": ["-"],
        }
        Rows=pd.concat((Rows,pd.DataFrame(new_row)))
    

    if saveresults:
        saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG LDA


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:14<00:00, 14.24s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_18.pkl


##############################
 EEG SVC


100%|██████████| 1/1 [00:13<00:00, 13.84s/it]


There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_18.pkl


##############################
 EEG RandomForest


100%|██████████| 1/1 [00:19<00:00, 19.59s/it]

There are 0 duplicated items (based on all columns except Date, ROIs, Accuracy).
 Index of the duplicated items: []

Results saved at: ../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_18.pkl


In [22]:
tutti_fold_selezioni_regioni

[[2, 3, 25, 33, 46, 60, 64, 65],
 [2, 3, 25, 32, 33, 39, 46, 60],
 [2, 29, 32, 33, 46, 52, 60, 62],
 [2, 3, 33, 36, 46, 60, 64, 65],
 [2, 3, 25, 33, 39, 46, 60, 65]]

In [15]:
import pandas as pd
uu = pd.read_pickle("../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_18.pkl")

In [ ]:
uu.Accuracy.apply(np.mean)

0    0.552347
1    0.577600
2    0.601664
3    0.549554
4    0.573678
5    0.576530
Name: Accuracy, dtype: float64

In [ ]:
uu.Classifier

0             LDA
1             SVC
2    RandomForest
3             LDA
4             SVC
5    RandomForest
Name: Classifier, dtype: object

In [7]:
set(uu["Region Selection"])


{"Cohen's d effect size selection",
 'Global Selection - Cohen based',
 'Yes Important Regions - all'}

# DO NOT DELETE

In [ ]:
# Load model
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras
model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)
model.build(input_shape=(128,1))
model.load_weights("../_Deep_Learning/Models_trained/2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try/Split_1/model.weights.h5")

2026-02-03 12:01:33.529346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 12:01:33.607088: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 12:01:35.168641: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1770116496.951234 1977451 gpu_device.cc:2020] Created device /job:lo

8 128


In [ ]:
import numpy as np
zeri = np.zeros(shape=(3,128,1))
model.encoder(zeri)

2026-02-02 16:37:42.890048: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91800


<tf.Tensor: shape=(3, 16), dtype=float32, numpy=
array([[-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542]], dtype=float32)>